# Age-stratified cis-heritability and AlphaGenome intervals

Pipeline overview:

1. Inputs and reference tables
2. Donor / tissue prep: valid tissues, phenotype files, young/old donor ID lists, gene batches
3. Heritability estimates (GEMMA BSLMM, GCTA REML), young vs old comparisons, threshold sensitivity
4. CTPred model manifest
5. Predicted vs observed expression, split by heritability and pLI
6. AlphaGenome input intervals

## 0. Setup

In [1]:
import os
import math
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from IPython.display import clear_output

In [16]:
# ---- Paths -----------------------------------------------------------------
BASE      = r'R:\heritability'
DGTEX_DIR = r'R:\dGTEx'

TPM_DIR           = os.path.join(BASE, 'aGTEx_gene_TPMs_by_tissue')
FAM_EXP_DIR       = os.path.join(BASE, 'fam_files_exp')          # expression .fam files (from the aGTEx prep notebook)
PHENO_DIR         = os.path.join(BASE, 'pheno_files')
AGE_IDS_DIR       = os.path.join(BASE, 'age_ids')
IDS_DIR           = os.path.join(BASE, 'ids')
GENE_COL_MAP_DIR  = os.path.join(BASE, 'gene_column_mappings')
GEMMA_RUNS_DIR    = os.path.join(BASE, 'gemma_runs')
GEMMA_TEST_DIR    = os.path.join(BASE, 'gemma_test')
GEMMA_FRESH_DIR    = os.path.join(BASE, 'gemma_fresh')
REML_DIR          = os.path.join(BASE, 'reml_outputs')
H2_DIR            = os.path.join(BASE, 'h2_dfs_2')
PREDICTED_EXP_DIR = os.path.join(BASE, 'predicted_exp')

CTPRED_MODELS_DIR   = os.path.join(DGTEX_DIR, 'agtex-ctpred-models')
CLUSTER_CTPRED_DIR  = '/gpfs/data/im-lab/nas40t2/Vincent/dGTEx/agtex-ctpred-models'

# ---- Constants -------------------------------------------------------------
TPM_PREFIX = 'gene_tpm_2017-06-05_v8_'
TPM_SUFFIX = '.gct.gz'

VALID_TISSUES = [
    'adipose_subcutaneous', 'artery_tibial', 'esophagus_mucosa', 'esophagus_muscularis',
    'muscle_skeletal', 'nerve_tibial', 'skin_sun_exposed_lower_leg', 'whole_blood',
]
CHUNK_SIZE      = 400   # genes per GEMMA batch
N_GENE_BATCHES  = 44
GENE_BATCHES    = [f'gene_batch_{i + 1}' for i in range(N_GENE_BATCHES)]

# Steps that draw random donors or re-read thousands of result files are skipped
# when their output already exists, unless this is True.
OVERWRITE = False

In [3]:
def short_id(sample_id):
    """'GTEX-1117F-0226-SM-5GZZ7' -> 'GTEX-1117F'"""
    return '-'.join(sample_id.split('-')[:2])


def load_tpm(tissue):
    """aGTEx TPM table for a tissue (columns: Name, Description, one column per sample)."""
    path = os.path.join(TPM_DIR, f'{TPM_PREFIX}{tissue}{TPM_SUFFIX}')
    return pd.read_csv(path, sep='\t', skiprows=2, compression='gzip', index_col=0)


def write_ids(ids, path):
    """PLINK-style keep file: FID IID (identical here)."""
    pd.DataFrame({0: ids, 1: ids}).to_csv(path, sep=' ', header=None, index=False)

## 1. Inputs and reference tables

In [4]:
agtex_metadata = pd.read_table(os.path.join(BASE, 'agtex_metadata.txt'))
gene_tss       = pd.read_table(os.path.join(BASE, 'gene_tss_lookup.tsv'))   # gene_idx, ensembl_id, gene_name, ...
pli_df         = pd.read_csv(os.path.join(BASE, 'pLI.csv'))

# Donor sets for the age comparisons
young_40 = list(agtex_metadata.loc[agtex_metadata['AGE'] <= 40, 'SUBJID'])
old_60   = list(agtex_metadata.loc[agtex_metadata['AGE'] >= 60, 'SUBJID'])

## 2. Donor / tissue prep

Order: choose tissues with enough young and old donors → write phenotype files → sample young/old donor ID lists → split genes into GEMMA batches.

### 2.1 Tissues with ≥ 100 young (≤ 40) and ≥ 100 old (≥ 60) donors

In [ ]:
fam_files = os.listdir(FAM_EXP_DIR)

valid_fams = []
for file in fam_files:
    fam = pd.read_table(os.path.join(FAM_EXP_DIR, file), sep=' ', header=None)
    if fam[1].isin(young_40).sum() >= 100 and fam[1].isin(old_60).sum() >= 100:
        valid_fams.append(file)

pd.Series(valid_fams).to_csv(os.path.join(BASE, 'valid_fams1.txt'), sep='\t', index=False, header=None)

# The analysis uses the 8 tissues in VALID_TISSUES; check the computed list agrees
files_to_use = [f'{t}_agtex.fam' for t in VALID_TISSUES]
print('computed list matches VALID_TISSUES:', sorted(valid_fams) == sorted(files_to_use))

### 2.2 Phenotype files (FID, IID, one column per gene)

In [ ]:
os.makedirs(PHENO_DIR, exist_ok=True)

for file in files_to_use:
    tissue = file.split('_agtex.fam')[0]
    fam = pd.read_table(os.path.join(FAM_EXP_DIR, file), sep=' ', header=None)
    fam.drop([2, 3, 4, 5], axis=1).to_csv(
        os.path.join(PHENO_DIR, f'{tissue}_agtex_pheno.txt'), sep=' ', header=None, index=False)

### 2.3 Donor ID lists

In [ ]:
pheno_dfs = {t: pd.read_table(os.path.join(PHENO_DIR, f'{t}_agtex_pheno.txt'), sep=' ', header=None)
             for t in VALID_TISSUES}

In [ ]:
# All donors per tissue
os.makedirs(IDS_DIR, exist_ok=True)
for t, df in pheno_dfs.items():
    write_ids(list(df[0]), os.path.join(IDS_DIR, f'{t}_all_ids.txt'))

In [ ]:
# 100 random young (<= 40) and 100 random old (>= 60) donors per tissue.
# Random draw (unseeded): existing files are kept unless OVERWRITE = True.
for t, df in pheno_dfs.items():
    out_dir = os.path.join(AGE_IDS_DIR, t)
    os.makedirs(out_dir, exist_ok=True)
    young_path = os.path.join(out_dir, 'agtex_young_ids_40.txt')
    old_path   = os.path.join(out_dir, 'agtex_old_ids_60.txt')
    if os.path.exists(young_path) and os.path.exists(old_path) and not OVERWRITE:
        print(f'{t}: exists, skipping')
        continue

    young_ids = list(df[df[0].isin(young_40)][0])
    old_ids   = list(df[df[0].isin(old_60)][0])
    write_ids(random.sample(young_ids, k=100), young_path)
    write_ids(random.sample(old_ids,   k=100), old_path)

Threshold-sensitivity donor sets (adipose only): young = AGE ≤ threshold, old = AGE > threshold, for thresholds 40 / 50 / 60.
These go to `age_ids/adipose/` (note: not `adipose_subcutaneous`).

In [ ]:
adipose_donors = pheno_dfs['adipose_subcutaneous'][0]
md_adipose     = agtex_metadata[agtex_metadata['SUBJID'].isin(adipose_donors)]

out_dir = os.path.join(AGE_IDS_DIR, 'adipose')
os.makedirs(out_dir, exist_ok=True)

for thr in (40, 50, 60):
    young_path = os.path.join(out_dir, f'agtex_young_ids_{thr}.txt')
    old_path   = os.path.join(out_dir, f'agtex_old_ids_{thr}.txt')
    if os.path.exists(young_path) and os.path.exists(old_path) and not OVERWRITE:
        print(f'threshold {thr}: exists, skipping')
        continue

    young_ids = list(md_adipose.loc[md_adipose['AGE'] <= thr, 'SUBJID'])
    old_ids   = list(md_adipose.loc[md_adipose['AGE'] >  thr, 'SUBJID'])
    write_ids(random.sample(young_ids, k=100), young_path)
    write_ids(random.sample(old_ids,   k=100), old_path)

### 2.4 Gene column mapping and GEMMA batches

The mapping from the aGTEx prep notebook numbers genes from 2 (the `.fam` has its own placeholder phenotype column).
The phenotype files above drop that column, so numbering is reset to start at 1. This is recomputed rather than decremented, so re-running is safe.

In [6]:
gene_col_map = pd.read_table(os.path.join(BASE, 'gene_column_mapping.txt'), header=None)   # gene, column
assert (gene_col_map[1].diff().dropna() == 1).all(), 'gene columns are not contiguous'

gene_col_map[1] = np.arange(1, len(gene_col_map) + 1)
gene_col_map.to_csv(os.path.join(BASE, 'gene_column_mapping.txt'), sep='\t', index=False, header=None)

In [ ]:
n_chunks = math.ceil(len(gene_col_map) / CHUNK_SIZE)
assert n_chunks == N_GENE_BATCHES, f'{n_chunks} chunks, expected {N_GENE_BATCHES}'

os.makedirs(GENE_COL_MAP_DIR, exist_ok=True)
for i in range(n_chunks):
    chunk = gene_col_map.iloc[i * CHUNK_SIZE:(i + 1) * CHUNK_SIZE]
    chunk.to_csv(os.path.join(GENE_COL_MAP_DIR, f'gene_col_map_{i + 1}.txt'),
                 header=None, index=False, sep='\t')

In [ ]:
# Output directories for GEMMA runs (add tissues here as they are started)
for tissue in ['adipose_subcutaneous']:
    for batch in GENE_BATCHES:
        os.makedirs(os.path.join(GEMMA_RUNS_DIR, tissue, batch), exist_ok=True)

In [ ]:
# Tissue list consumed by the dGTEx pipeline
pd.DataFrame({'agtexContext': VALID_TISSUES}).to_csv(
    os.path.join(DGTEX_DIR, 'DATA', 'tissue_mapping_intersect_2.txt'), sep=' ', index=False)

### 2.5 Inputs for the GEMMA threshold test (adipose)

In [ ]:
test_fam = pd.read_table(os.path.join(GEMMA_TEST_DIR, files_to_use[0]), sep=' ', header=None)
test_tissue = VALID_TISSUES[0]

test_fam.to_csv(os.path.join(GEMMA_TEST_DIR, f'{test_tissue}_agtex_master.fam'), sep=' ', header=None, index=False)
test_fam.drop([2, 3, 4, 5], axis=1).to_csv(
    os.path.join(GEMMA_TEST_DIR, f'{test_tissue}_agtex_pheno.txt'), sep=' ', header=None, index=False)

## 3. Heritability estimates

### 3.1 GEMMA (BSLMM) h² per gene, young vs old

Each gene has `<gene>_young` and `<gene>_old` run directories under `gemma_runs/<tissue>/gene_batch_N/`.
h² is the median of the `h ` column of `result.hyp.txt`. A gene is kept only if both runs can be read.

In [12]:
def read_hyp_median(path, reset_index=False):
    """Median of the 'h ' column of a GEMMA result.hyp.txt."""
    df = pd.read_table(path)
    if reset_index:
        df = df.reset_index()
    return np.median(df['h '])


def gemma_h2_df(tissue):
    """Per-gene young / old h2 for a tissue. Returns (DataFrame, genes that failed)."""
    rows, failed = [], []
    for n, batch in enumerate(GENE_BATCHES, start=1):
        clear_output(wait=True)
        print(f'{tissue}: {batch} ({n}/{len(GENE_BATCHES)})')
        batch_dir = os.path.join(GEMMA_FRESH_DIR, tissue, batch)
        genes = [d[:-len('_young')] for d in os.listdir(batch_dir) if d.endswith('_young')]
        for gene in genes:
            try:
                h_y = read_hyp_median(os.path.join(batch_dir, f'{gene}_young', 'result.hyp.txt'))
                h_o = read_hyp_median(os.path.join(batch_dir, f'{gene}_old',   'result.hyp.txt'))
            except Exception:
                failed.append(gene)
                continue
            rows.append((gene, h_y, h_o))
    return pd.DataFrame(rows, columns=['gene', 'h2_y', 'h2_o']).dropna(), failed


def count_finished_runs(tissue):
    """Number of young / old run directories present across all batches (progress check)."""
    counts = {'young': 0, 'old': 0}
    for batch in GENE_BATCHES:
        for d in os.listdir(os.path.join(GEMMA_FRESH_DIR, tissue, batch)):
            if 'agtex' in d:
                continue
            for group in counts:
                if d.endswith(f'_{group}'):
                    counts[group] += 1
    return counts

In [15]:
count_finished_runs('adipose_subcutaneous')

{'young': 9254, 'old': 9236}

In [17]:
# Tissues with finished runs. Skips tissues whose h2 table already exists unless OVERWRITE = True.
H2_TISSUES = ['adipose_subcutaneous', 'artery_tibial', 'esophagus_mucosa',
              'esophagus_muscularis', 'nerve_tibial']

os.makedirs(H2_DIR, exist_ok=True)
for tissue in H2_TISSUES:
    out = os.path.join(H2_DIR, f'{tissue}_h2_df.tsv')
    if os.path.exists(out) and not OVERWRITE:
        print(f'{tissue}: exists, skipping')
        continue
    df, failed = gemma_h2_df(tissue)
    df.to_csv(out, sep='\t', index=False)
    print(f'{tissue}: {len(df)} genes, {len(failed)} failed')

nerve_tibial: gene_batch_44 (44/44)
nerve_tibial: 0 genes, 0 failed


### 3.2 GCTA (REML) h² per gene, adipose

In [ ]:
def read_gcta_h2(tissue, group):
    """h2 (V(G)/Vp) from each gene_<n>.hsq under reml_outputs/<tissue>/<group>. gene_idx is 1-based."""
    d = os.path.join(REML_DIR, tissue, group)
    rows = []
    for f in os.listdir(d):
        if not (f.startswith('gene_') and f.endswith('.hsq')):
            continue
        gene_idx = int(f.split('gene_')[1].split('.hsq')[0])
        try:
            rows.append((gene_idx, pd.read_table(os.path.join(d, f)).loc[3]['Variance']))
        except Exception:
            continue
    return pd.DataFrame(rows, columns=['gene_idx', 'h2']).sort_values('gene_idx').reset_index(drop=True)


gene_labels = gene_tss[['gene_idx', 'ensembl_id', 'gene_name']]
gcta_h2_young_df = pd.merge(gene_labels, read_gcta_h2('adipose_subcutaneous', 'young'))
gcta_h2_old_df   = pd.merge(gene_labels, read_gcta_h2('adipose_subcutaneous', 'old'))

os.makedirs(H2_DIR, exist_ok=True)
gcta_h2_young_df.to_csv(os.path.join(H2_DIR, 'gcta_h2_young_df.tsv'), sep='\t', index=False)
gcta_h2_old_df.to_csv(os.path.join(H2_DIR, 'gcta_h2_old_df.tsv'),   sep='\t', index=False)

In [ ]:
# Distribution of estimates restricted to the valid range [0, 1]
in_range = lambda s: s[(s >= 0) & (s <= 1)]

plt.boxplot([in_range(gcta_h2_old_df['h2']), in_range(gcta_h2_young_df['h2'])], tick_labels=['old', 'young'])
plt.ylabel('GCTA h2')
plt.title('Adipose Subcutaneous')
plt.show()

gcta_h2_young_df['h2'].describe()

### 3.3 BSLMM vs GCTA (adipose, young)

In [ ]:
h2_df_ad = pd.read_table(os.path.join(H2_DIR, 'adipose_subcutaneous_h2_df.tsv'))   # gene, h2_y, h2_o

gcta_young_valid = gcta_h2_young_df[np.abs(gcta_h2_young_df['h2']) <= 1]
method_df_young = pd.merge(h2_df_ad.rename(columns={'gene': 'gene_name'})[['gene_name', 'h2_y']],
                           gcta_young_valid)

plt.scatter(method_df_young['h2_y'], method_df_young['h2'], alpha=0.05)
plt.xlabel('BSLMM-calculated h2')
plt.ylabel('GCTA-calculated h2')
plt.title('Adipose Subcutaneous, young')
plt.show()

print('Pearson: ', stats.pearsonr(method_df_young['h2_y'], method_df_young['h2']))
print('Spearman:', stats.spearmanr(method_df_young['h2_y'], method_df_young['h2']))

### 3.4 Young vs old h² (BSLMM)

In [ ]:
h2_dfs = {}
for t in H2_TISSUES:
    path = os.path.join(H2_DIR, f'{t}_h2_df.tsv')
    if os.path.exists(path):
        h2_dfs[t] = pd.read_table(path)


def summarize_h2(df):
    y, o = df['h2_y'], df['h2_o']
    return {
        'n_genes': len(df),
        'median_h2_young': y.median(),
        'median_h2_old': o.median(),
        'mean_abs_diff': np.abs(y - o).mean(),
        'wilcoxon_p': stats.wilcoxon(y, o).pvalue,      # paired, young vs old
        'pearson_r': stats.pearsonr(y, o)[0],
    }


pd.DataFrame({t: summarize_h2(df) for t, df in h2_dfs.items()}).T

In [ ]:
# Per tissue: histogram, young-vs-old scatter, and matched-percentile (QQ-style) plot
percentiles = np.arange(1, 100, 0.5)

for tissue, df in h2_dfs.items():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(df['h2_y'], alpha=0.25, label='young')
    axes[0].hist(df['h2_o'], alpha=0.25, label='old')
    axes[0].set(xlabel='h2', ylabel='Gene count')
    axes[0].legend()

    axes[1].scatter(df['h2_y'], df['h2_o'], alpha=0.3)
    axes[1].set(xlabel='Young h2', ylabel='Old h2')

    axes[2].scatter(np.percentile(df['h2_y'].dropna(), percentiles),
                    np.percentile(df['h2_o'].dropna(), percentiles))
    axes[2].plot([0, 1], [0, 1], color='red')
    axes[2].set(xlabel='Young h2 (percentiles)', ylabel='Old h2 (percentiles)')

    fig.suptitle(f'Gene Expression Cis-heritability ({tissue})')
    plt.show()

### 3.5 Threshold sensitivity (adipose)

Medians of `h` for young / old runs at age thresholds 40, 50, 60, read from `gemma_test/thresh_<N>/<gene>_<group>/`.

**Assumption:** the gene directories are taken from the listing of `thresh_40` (the original notebook used a `h2_dirs` variable that was defined in a cell not present in the file).
These test outputs are read with `reset_index()` before selecting `h `, as in the original.

In [ ]:
THRESHOLDS = (40, 50, 60)

h2_dirs   = os.listdir(os.path.join(GEMMA_TEST_DIR, 'thresh_40'))
dirs_by_group = {
    'young': [d for d in h2_dirs if d.endswith('_young')],
    'old':   [d for d in h2_dirs if d.endswith('_old')],
}

thresh_medians, thresh_failed = {}, {}
for thr in THRESHOLDS:
    for group, dirs in dirs_by_group.items():
        medians, failed = [], []
        for d in dirs:
            path = os.path.join(GEMMA_TEST_DIR, f'thresh_{thr}', d, 'result.hyp.txt')
            try:
                medians.append(read_hyp_median(path, reset_index=True))
            except Exception:
                failed.append(d)
        thresh_medians[(thr, group)] = medians
        thresh_failed[(thr, group)] = failed

pd.DataFrame(
    [{'threshold': thr, 'group': group, 'n': len(m), 'n_failed': len(thresh_failed[(thr, group)]),
      'q25': np.percentile(m, 25), 'median': np.percentile(m, 50), 'q75': np.percentile(m, 75)}
     for (thr, group), m in thresh_medians.items()]
).round(3)

In [ ]:
plt.boxplot([pd.Series(thresh_medians[(50, 'young')]).dropna(),
             pd.Series(thresh_medians[(50, 'old')]).dropna()], tick_labels=['young', 'old'])
plt.ylabel('GEMMA h (median)')
plt.title('Adipose, threshold 50')
plt.show()

Format check on one result file. The `result.hyp.txt` header appears to be one column short, so pandas moves the first column into the index;
the cell below relabels the columns the way the original notebook did. The readers above use `['h ']` directly, so it is worth confirming which column that is.

In [ ]:
result_hyp = pd.read_table(os.path.join(GEMMA_TEST_DIR, 'thresh_40', 'KLHL17_old', 'result.hyp.txt'))
hyp_cols = list(result_hyp.columns) + ['drop']
result_hyp = result_hyp.reset_index()
result_hyp.columns = hyp_cols
result_hyp = result_hyp.drop('drop', axis=1)

result_hyp[' pve '].sort_values()

## 4. CTPred model manifest

In [ ]:
ctpred_tissues = os.listdir(CTPRED_MODELS_DIR)

contexts, paths = [], []
for tissue in ctpred_tissues:
    models = [m for m in os.listdir(os.path.join(CTPRED_MODELS_DIR, tissue)) if 'chunk' not in m]
    for m in models:
        donor = m.split('.')[3]
        contexts.append(f'{tissue}_{donor}')
        paths.append(f'{CLUSTER_CTPRED_DIR}/{tissue}/{m}')

pd.DataFrame({'context': contexts, 'path': paths}).to_csv(
    os.path.join(DGTEX_DIR, 'ctpred.agtex_models.tsv'), sep='\t', index=False)

## 5. Predicted vs observed expression (adipose), split by heritability and pLI

Two prediction sources are compared with observed TPM. In both, "predictability" is the per-donor Spearman correlation between predicted and observed expression across genes (or loci).
Each subsection reloads the TPM table, since the processing differs.

In [ ]:
def per_donor_spearman(predicted, observed, donors, key):
    """Spearman rho / p between predicted and observed values across `key` rows, one per donor column."""
    rows = []
    for d in donors:
        m = pd.merge(predicted[[key, d]], observed[[key, d]], on=key,
                     suffixes=('_pred', '_obs')).dropna()
        res = stats.spearmanr(m[f'{d}_pred'], m[f'{d}_obs'])
        rows.append({'donor': d, 'rho': res.statistic, 'p': res.pvalue})
    return pd.DataFrame(rows)

### 5.1 Locus-based predictions (dGTEx output)

Observed TPM is mapped to prediction loci, merged with BSLMM young h² and pLI, and split at the median. 

In [ ]:
predicted_exp_ad = pd.read_table(os.path.join(DGTEX_DIR, 'OUTPUTS', 'predicted_expression',
                                              'reference.agtex.adipose_subcutaneous.all.models.txt'))
predicted_exp_ad.columns = list(predicted_exp_ad.columns[:4]) + [c.split('.')[2] for c in predicted_exp_ad.columns[4:]]

locus_gene_map = (pd.read_table(os.path.join(DGTEX_DIR, 'DATA', 'prediction_locus_gene_map.tsv'))
                  .rename(columns={'prediction_locus': 'locus'}))

In [ ]:
adipose_tpm = load_tpm('adipose_subcutaneous')
adipose_tpm['Name'] = adipose_tpm['Name'].str.split('.').str[0]      # drop Ensembl version
adipose_tpm = pd.merge(locus_gene_map, adipose_tpm.rename(columns={'Name': 'gene_name'}))
adipose_tpm.columns = list(adipose_tpm.columns[:4]) + ['GTEX-' + c.split('-')[1] for c in adipose_tpm.columns[4:]]

adipose_tpm_2 = pd.merge(pd.merge(adipose_tpm[['locus', 'Description']].rename(columns={'Description': 'gene'}),
                                  h2_df_ad),
                         adipose_tpm)
adipose_tpm_3 = adipose_tpm_2[adipose_tpm_2['locus'].isin(predicted_exp_ad['locus'])].reset_index(drop=True)
adipose_tpm_4 = pd.merge(pli_df.rename(columns={'gene': 'Description'})[['Description', 'pLI']], adipose_tpm_3)

# median splits
high_h2 = adipose_tpm_3[adipose_tpm_3['h2_y'] >= np.median(adipose_tpm_3['h2_y'])]
low_h2  = adipose_tpm_3[adipose_tpm_3['h2_y'] <  np.median(adipose_tpm_3['h2_y'])]
high_pli = adipose_tpm_4[adipose_tpm_4['pLI'] >= np.median(adipose_tpm_4['pLI'])]
low_pli  = adipose_tpm_4[adipose_tpm_4['pLI'] <  np.median(adipose_tpm_4['pLI'])]

donors_ad = [c for c in adipose_tpm_4.columns if c.startswith('GTEX-')]

In [ ]:
pred_h2_low  = per_donor_spearman(predicted_exp_ad, low_h2,  donors_ad, key='locus')
pred_h2_high = per_donor_spearman(predicted_exp_ad, high_h2, donors_ad, key='locus')

plt.boxplot([pred_h2_low['rho'], pred_h2_high['rho']], tick_labels=['Low h2', 'High h2'])
plt.ylabel('Predictability')
plt.title('Adipose Subcutaneous')
plt.show()

print('Wilcoxon (low vs high h2):', stats.wilcoxon(pred_h2_low['rho'], pred_h2_high['rho']))
print('mean rho  low / high:', pred_h2_low['rho'].mean(), pred_h2_high['rho'].mean())

In [ ]:
pred_pli_low  = per_donor_spearman(predicted_exp_ad, low_pli,  donors_ad, key='locus')
pred_pli_high = per_donor_spearman(predicted_exp_ad, high_pli, donors_ad, key='locus')

plt.boxplot([pred_pli_low['rho'], pred_pli_high['rho']], tick_labels=['Low pLI', 'High pLI'])
plt.ylabel('Predictability')
plt.title('Adipose Subcutaneous')
plt.show()

print('Wilcoxon (low vs high pLI):', stats.wilcoxon(pred_pli_low['rho'], pred_pli_high['rho']))
print('mean rho  low / high:', pred_pli_low['rho'].mean(), pred_pli_high['rho'].mean())

In [ ]:
# h2 by pLI group
for col, label in [('h2_y', 'Young h2'), ('h2_o', 'Old h2')]:
    plt.boxplot([low_pli[col], high_pli[col]], tick_labels=['Low pLI', 'High pLI'])
    plt.ylabel(label)
    plt.title('Adipose Subcutaneous')
    plt.show()

plt.hist(low_pli['h2_y'],  alpha=0.3, label='Low pLI')
plt.hist(high_pli['h2_y'], alpha=0.3, label='High pLI')
plt.xlabel('Young h2')
plt.legend()
plt.show()

### 5.2 Gene-based predictions (`predicted_exp/AdiposeSubcutaneous_predict.txt`)

Prediction columns are named by Ensembl ID and converted to gene symbols using the TPM table; genes are split at the median young h².

In [ ]:
adipose_tpm = load_tpm('adipose_subcutaneous')

# Ensembl ID (version stripped) -> gene symbol
conversion_dict = dict(zip(adipose_tpm['Name'].str.split('.').str[0], adipose_tpm['Description']))

predicted_exp_ad = pd.read_table(os.path.join(PREDICTED_EXP_DIR, 'AdiposeSubcutaneous_predict.txt'))
ensgs = [c.split('AdiposeSubcutaneous_')[1].split('_chr')[0] for c in predicted_exp_ad.columns[2:]]
predicted_exp_ad.columns = ['FID', 'IID'] + [conversion_dict.get(e, e) for e in ensgs]

# observed TPM with donor-level column names
adipose_tpm.columns = list(adipose_tpm.columns[:2]) + [short_id(c) for c in adipose_tpm.columns[2:]]

In [ ]:
def predicted_by_gene(predicted_exp, genes):
    """Wide table (rows = genes, columns = donor IIDs) of predicted expression for `genes`."""
    gene_set = set(genes)
    cols = [c for c in predicted_exp.columns if c in gene_set]
    wide = predicted_exp[['IID'] + cols].set_index('IID').T
    wide.index.name = 'Description'
    return wide.reset_index()


median_h2 = np.median(h2_df_ad['h2_y'])
high_h2_genes = list(h2_df_ad.loc[h2_df_ad['h2_y'] >= median_h2, 'gene'])
low_h2_genes  = list(h2_df_ad.loc[h2_df_ad['h2_y'] <  median_h2, 'gene'])

obs_high = adipose_tpm[adipose_tpm['Description'].isin(high_h2_genes)]
obs_low  = adipose_tpm[adipose_tpm['Description'].isin(low_h2_genes)]
pred_high = predicted_by_gene(predicted_exp_ad, high_h2_genes)
pred_low  = predicted_by_gene(predicted_exp_ad, low_h2_genes)

shared_donors = sorted(set(predicted_exp_ad['IID']) & set(adipose_tpm.columns[2:]))

In [ ]:
res_high = per_donor_spearman(pred_high, obs_high, shared_donors, key='Description')
res_low  = per_donor_spearman(pred_low,  obs_low,  shared_donors, key='Description')

pred_df_ad = pd.DataFrame({
    'donor': shared_donors,
    'high_h2_rho': res_high['rho'], 'high_h2_p': res_high['p'],
    'low_h2_rho':  res_low['rho'],  'low_h2_p':  res_low['p'],
})

plt.hist(pred_df_ad['high_h2_rho'], alpha=0.3, label='top 50% h2')
plt.hist(pred_df_ad['low_h2_rho'],  alpha=0.3, label='bottom 50% h2')
plt.xlabel('Genetic Predictability')
plt.ylabel('Donor Count')
plt.title('h2 vs. genetic predictability; adipose subcutaneous')
plt.legend()
plt.show()

## 6. AlphaGenome input intervals

Each interval in `sequences_human.bed` is re-centered to a 2²⁰ bp (1,048,576) window around its midpoint.

In [ ]:
AG_WINDOW = 2 ** 20

chrom_lengths = {
    'chr1': 248956422, 'chr2': 242193529, 'chr3': 198295559, 'chr4': 190214555, 'chr5': 181538259, 'chr6': 170805979,
    'chr7': 159345973, 'chr8': 145138636, 'chr9': 138394717, 'chr10': 133797422, 'chr11': 135086622, 'chr12': 133275309,
    'chr13': 114364328, 'chr14': 107043718, 'chr15': 101991189, 'chr16': 90338345, 'chr17': 83257441, 'chr18': 80373285,
    'chr19': 58617616, 'chr20': 64444167, 'chr21': 46709983, 'chr22': 50818468, 'chrX': 156040895, 'chrY': 57227415,
}

interval_df = pd.read_table('sequences_human.bed', header=None)
interval_df.columns = ['chrom', 'start', 'end', 'fold']
interval_df = interval_df.sort_values(by=['chrom', 'start'])

interval_df['midpoint'] = interval_df['start'] + (interval_df['end'] - interval_df['start']) // 2
interval_df['ag_start'] = interval_df['midpoint'] - AG_WINDOW // 2
interval_df['ag_end']   = interval_df['midpoint'] + AG_WINDOW // 2
interval_df = interval_df.reset_index(drop=True)
interval_df

In [ ]:
# How many windows run off either end of their chromosome?
chrom_len = interval_df['chrom'].map(chrom_lengths)
outside = (interval_df['ag_start'] < 0) | (interval_df['ag_end'] > chrom_len)
print(f'{outside.sum()} of {len(interval_df)} windows extend past a chromosome boundary')